<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

## 🎯 创建一个 deployment（部署）

**一句话定位**：把 module-5 做的 `task_maistro` agent 打包成一个可远程调用的 LangGraph 服务。

</div>

<div class="dark-cyan" style="background:#0f2729; color:#a5f3fc; padding:10px 24px; border-left:4px solid #22d3ee; border-radius:4px; width:97%;"><style>.dark-cyan strong{color:#fbbf24;}</style>

**🆕 2026 更新**：2025 年 10 月起，**LangGraph Platform**（托管部署服务）改名为 **LangSmith Deployment**；**LangGraph Studio**（调试 IDE）改名为 **LangSmith Studio**。这纯粹是品牌/命名调整——API、SDK、已有部署全部照常工作，不需要改任何代码。LangSmith 现在统一包含三块：Observability（可观测性）、Evaluation（评估）、Deployment（部署）。本笔记后面提到的 "LangGraph Platform" 均可理解为今天的 LangSmith Deployment。

</div>

## 代码结构

创建一个 LangGraph Platform（= LangSmith Deployment）部署需要 4 样东西：

* **配置文件** `langgraph.json`——[LangGraph API 配置文件](https://docs.langchain.com/langsmith/application-structure#configuration-file)
* **graph 代码**——实现应用逻辑，如 `task_maistro.py`
* **依赖文件** `requirements.txt`
* **环境变量**——通过 `.env` 或 `docker-compose.yml` 提供

这些都已经在 `module-6/deployment` 目录里准备好了。

## CLI

[LangGraph CLI](https://docs.langchain.com/langsmith/cli) 是用来创建 LangGraph 部署的命令行工具（这是开源框架自带的工具，名字没有跟着品牌改名）。

### 自托管部署（self-hosted deployment）

#### 第一步：为 LangGraph Server 构建 Docker 镜像

用 langgraph CLI 把 graph 和依赖打包成一个 [Docker](https://docs.docker.com/engine/install/) 镜像（镜像是容器的模板，装好了运行所需的代码和依赖）：

```
$ cd module-6/deployment
$ langgraph build -t my-image
```

#### 第二步：准备 Redis 和 PostgreSQL

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

LangGraph Server 依赖 **Redis**（任务队列/发布订阅）和 **PostgreSQL**（持久化存储）。如果你已经有现成的 Redis/Postgres 实例，可以单独起 LangGraph Server 容器，把地址通过环境变量传进去：

</div>

```
docker run \
    --env-file .env \
    -p 8123:8000 \
    -e REDIS_URI="foo" \
    -e DATABASE_URI="bar" \
    -e LANGSMITH_API_KEY="baz" \
    my-image
```

也可以直接用项目提供的 `docker-compose.yml`，一次性拉起三个容器：

* `langgraph-redis` —— 官方 Redis 镜像
* `langgraph-postgres` —— 官方 Postgres 镜像
* `langgraph-api` —— 你刚才构建的镜像

把 `docker-compose-example.yml` 复制一份，填入：

* `IMAGE_NAME`（例如 `my-image`）
* `LANGSMITH_API_KEY`
* `OPENAI_API_KEY`

然后启动部署：

```
$ cd module-6/deployment
$ docker compose up
```

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：一个 LangGraph 部署 = graph 代码 + 配置文件 + Docker 镜像 + Redis/Postgres，`docker compose up` 之后它就是一个能被远程调用（创建 assistant、thread、run）的 HTTP 服务。

</div>